Neural Style Transfer implementation based on Gatys et al. (2015)
Inspired by the official PyTorch tutorial
Includes Optuna for automated Bayesian hyperparameter tuning.


Understanding the loss outputs is critical for objective evaluation:

1. Raw Feature Magnitudes:
   - VGG-19 extracts features after ReLU activations. These values can be large.
   - Raw Content Loss (MSE of conv4_2) usually falls between [1.0, 50.0].
   - Raw Style Loss (MSE of Gram Matrices) is much smaller because Gram
     matrices are normalized by (Channels * Height * Width). It typically falls
     between [1e-6, 1e-4].

2. Optuna Objective Score (mock_perceptual_score):
   - To prevent Optuna from cheating by simply dropping Beta to 0 to minimize total loss, evaluate the final render using UNWEIGHTED losses.
   - Because raw style loss is so much smaller than content loss, apply a 1e5 scalar to the style error to put them on the same mathematical level.
   - A healthy Optuna score usually ranges from [5.0 to 30.0]. Lower is better.

3. Weighted Loss (Printed during optimization):
   - Alpha typically ranges from [0.01 to 100].
   - Beta typically ranges from [10,000 to 10,000,000].
   - The printed losses are scaled by these weights.
   
   Intended outcome: Style Loss dropping rapidly in the first 50 steps, while Content Loss slowly creeps up
     as the image distorts to accommodate the style transfer.

In [ ]:
import os
import csv
import glob
import time
import json
import gc
import shutil
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torchvision.models import VGG19_Weights # Fixes deprecation warnings
from PIL import Image
import optuna
import lpips
from torchmetrics.image import StructuralSimilarityIndexMeasure

# Silence Optuna to prevent terminal flooding during batch processing
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ==========================================
# 0. MOUNT GOOGLE DRIVE (ROBUST)
# ==========================================
try:
    from google.colab import drive
    # Clean up ghost directory if it exists before mounting
    if os.path.exists('/content/drive') and not os.path.ismount('/content/drive'):
        shutil.rmtree('/content/drive')
    drive.mount('/content/drive', force_remount=True)
    print("-> Google Drive mounted successfully.")
except Exception as e:
    print(f"-> WARNING: Not running in Colab or mount failed. {e}")

# ==========================================
# 1. CONFIGURATION & IDEMPOTENT SETUP
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Targeted pairs for evaluation benchmark
TARGET_CONTENTS = ["rim1024", "angel1024", "mountains1024", "daisy1024", "berries1024"]
TARGET_STYLES = ["starrynight", "la_muse", "feathers", "wave", "composition_vii", "mosaic"]

IMG_SIZE = 256
NUM_RUNS = 3       # Repeated runs per pair for statistical variance
TRIAL_STEPS = 50   # Steps for Bayesian sampler
FINAL_STEPS = 300  # Steps for final generation

# System paths
BASE_DIR = "/content/drive/MyDrive/NST_Datasets"
CONTENT_DIR = f"{BASE_DIR}/content_subset/images/nprgeneral"
STYLE_DIR = f"{BASE_DIR}/style_subset/images/style_targets"
OUTPUT_DIR = f"{BASE_DIR}/Output_subset/Gatys"
CSV_PATH = os.path.join(OUTPUT_DIR, "gatys_evaluation_results.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "saliency"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "convergence"), exist_ok=True)

# Idempotency: Track completed iterations to allow safe restarts
completed_pairs = set()
csv_headers = [
    "Content_Image", "Style_Image", "Run_Index", "Output_Image",
    "Optimized_Alpha", "Optimized_Beta", "Optimized_LR",
    "SSIM_Content", "LPIPS_Content", "Isolated_Gram_Style", "Inference_Latency_Sec"
]

# Create CSV or load existing progress
if os.path.exists(CSV_PATH):
    with open(CSV_PATH, mode='r') as f:
        reader = csv.reader(f)
        next(reader, None) # Skip header
        for row in reader:
            if len(row) >= 4:
                completed_pairs.add(row[3]) # Output_Image filename acts as unique hash
else:
    with open(CSV_PATH, mode='w', newline='') as f:
        csv.writer(f).writerow(csv_headers)

# ==========================================
# 2. METRICS & IMAGE UTILITIES
# ==========================================
def load_image(image_path, max_size=IMG_SIZE, shape=None):
    """Loads image, enforces exact resolution, and normalizes to ImageNet standards."""
    image = Image.open(image_path).convert('RGB')
    if max_size:
        size = max_size if max(image.size) > max_size else max(image.size)
        if shape is not None:
            size = shape
        transform = transforms.Compose([
            transforms.Resize(size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    return transform(image).unsqueeze(0).to(device)

def save_image(tensor, save_path):
    """Reverses ImageNet normalization and clamps pixel values to prevent artifacts."""
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(device)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(device)
    image_01 = (tensor.detach() * std + mean).clamp(0, 1)
    transforms.ToPILImage()(image_01.squeeze(0).cpu()).save(save_path)

def save_saliency_map(target_tensor, save_path):
    """Extracts absolute gradient magnitudes to visualize network spatial attention."""
    if target_tensor.grad is not None:
        saliency, _ = torch.max(target_tensor.grad.data.abs(), dim=1)
        saliency = (saliency - saliency.min()) / (saliency.max() - saliency.min() + 1e-8)
        transforms.ToPILImage()(saliency.squeeze(0).cpu()).save(save_path)

# ==========================================
# 3. VGG CRITIC (Architecture)
# ==========================================
class VGGNet(nn.Module):
    def __init__(self):
        super(VGGNet, self).__init__()
        # Updated to resolve deprecation warnings
        self.vgg = models.vgg19(weights=VGG19_Weights.IMAGENET1K_V1).features
        self.chosen_features = {
            '0': 'conv1_1', '5': 'conv2_1', '10': 'conv3_1',
            '19': 'conv4_1', '21': 'conv4_2', '28': 'conv5_1'
        }
        for param in self.vgg.parameters():
            param.requires_grad = False

    def forward(self, x):
        features = {}
        for name, layer in self.vgg._modules.items():
            x = layer(x)
            if name in self.chosen_features:
                features[self.chosen_features[name]] = x
        return features

def gram_matrix(tensor):
    b, c, h, w = tensor.size()
    features = tensor.view(c, h * w)
    return torch.mm(features, features.t()) / (c * h * w)

def calculate_isolated_style_loss(output_tensor, style_tensor, model):
    """Calculates unweighted, raw Gram MSE strictly for cross-architecture comparison."""
    with torch.no_grad():
        out_features = model(output_tensor)
        style_features = model(style_tensor)
        style_layers = ['conv1_1', 'conv2_1', 'conv3_1', 'conv4_1', 'conv5_1']

        total_gram_mse = 0.0
        for layer in style_layers:
            total_gram_mse += nn.MSELoss()(gram_matrix(out_features[layer]), gram_matrix(style_features[layer])).item()
    return total_gram_mse

# ==========================================
# 4. EXECUTION PIPELINE
# ==========================================
def main():
    # Filter files strictly against targeted basenames
    all_content_files = sorted(glob.glob(os.path.join(CONTENT_DIR, '*.[jp][pn]g')))
    all_style_files = sorted(glob.glob(os.path.join(STYLE_DIR, '*.[jp][pn]g')))

    content_files = [f for f in all_content_files if os.path.splitext(os.path.basename(f))[0] in TARGET_CONTENTS]
    style_files = [f for f in all_style_files if os.path.splitext(os.path.basename(f))[0] in TARGET_STYLES]

    print(f"Targeting {len(content_files)} Content images and {len(style_files)} Style images.")
    if not content_files or not style_files:
        raise ValueError("Filtered target arrays are empty. Check dataset paths and filenames.")

    # Initialize metrics
    model = VGGNet().to(device).eval()
    lpips_metric = lpips.LPIPS(net='vgg', verbose=False).to(device)
    ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)

    for content_path in content_files:
        c_name = os.path.splitext(os.path.basename(content_path))[0]
        for style_path in style_files:
            s_name = os.path.splitext(os.path.basename(style_path))[0]

            # --- LOOP: Repeated Runs for Statistical Variance ---
            for run_idx in range(1, NUM_RUNS + 1):
                output_filename = f"{c_name}_{s_name}_run{run_idx}.jpg"

                # Idempotent skip check
                if output_filename in completed_pairs:
                    print(f"Skipping (Already Processed): {output_filename}")
                    continue

                print(f"Processing: {output_filename}")

                content_img = load_image(content_path)
                style_img = load_image(style_path, shape=[content_img.size(2), content_img.size(3)])

                # Extract priors
                content_features = model(content_img)
                style_features = model(style_img)
                style_grams = {l: gram_matrix(style_features[l]).detach() for l in style_features}
                style_weights = {'conv1_1': 1.0, 'conv2_1': 0.8, 'conv3_1': 0.5, 'conv4_1': 0.3, 'conv5_1': 0.1}

                # --- PHASE 1: Optuna Optimization ---
                def objective(trial):
                    alpha = trial.suggest_float("alpha", 1e-2, 1e2, log=True)
                    beta = trial.suggest_float("beta", 1e4, 1e7, log=True)
                    lr = trial.suggest_float("lr", 0.1, 2.0, log=True)

                    t_img = content_img.clone().requires_grad_(True)
                    opt = optim.LBFGS([t_img], lr=lr)

                    # 50 step optimisation loop
                    for _ in range(TRIAL_STEPS):
                        def closure():
                            opt.zero_grad()
                            t_feat = model(t_img)
                            c_loss = nn.MSELoss()(t_feat['conv4_2'], content_features['conv4_2'].detach())
                            s_loss = sum([style_weights[l] * nn.MSELoss()(gram_matrix(t_feat[l]), style_grams[l]) for l in style_weights])
                            loss = alpha * c_loss + beta * s_loss
                            loss.backward()
                            return loss
                        opt.step(closure)

                    # evaluate unweighted tensor errors
                    with torch.no_grad():
                        f_feat = model(t_img)
                        c_err = nn.MSELoss()(f_feat['conv4_2'], content_features['conv4_2'].detach()).item()
                        s_err = sum([(style_weights[l] * nn.MSELoss()(gram_matrix(f_feat[l]), style_grams[l])).item() for l in style_weights])

                    # Mock perceptual score balances raw distances
                    return c_err + (s_err * 1e5)

                study = optuna.create_study(direction="minimize")
                study.optimize(objective, n_trials= NUM_RUNS)

                # --- PHASE 2: Inference & Convergence Tracking ---
                target_img = content_img.clone().requires_grad_(True)
                optimizer = optim.LBFGS([target_img], lr=study.best_params["lr"])
                loss_history = []

                # Precision Latency tracking (Excludes I/O overhead)
                inf_start = time.perf_counter()

                step_idx = [0]
                while step_idx[0] < FINAL_STEPS:
                    def closure():
                        optimizer.zero_grad()
                        t_feat = model(target_img)
                        c_loss = nn.MSELoss()(t_feat['conv4_2'], content_features['conv4_2'].detach())
                        s_loss = sum([style_weights[l] * nn.MSELoss()(gram_matrix(t_feat[l]), style_grams[l]) for l in style_weights])

                        total_loss = (study.best_params["alpha"] * c_loss) + (study.best_params["beta"] * s_loss)
                        total_loss.backward()

                        loss_history.append(total_loss.item())
                        step_idx[0] += 1
                        return total_loss

                    optimizer.step(closure)

                inf_latency = time.perf_counter() - inf_start

                # --- PHASE 3: Extraction & Metrological Evaluation ---
                output_path = os.path.join(OUTPUT_DIR, output_filename)
                save_image(target_img, output_path)

                # Save visual diagnostics
                save_saliency_map(target_img, os.path.join(OUTPUT_DIR, "saliency", f"sal_{output_filename}"))
                with open(os.path.join(OUTPUT_DIR, "convergence", f"conv_{c_name}_{s_name}_run{run_idx}.json"), 'w') as f:
                    json.dump(loss_history, f)

                # Format tensors to [0,1] and [-1,1] ranges for metrics
                mean, std = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1).to(device), torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1).to(device)
                c_01 = (content_img * std + mean).clamp(0,1)
                o_01 = (target_img.detach() * std + mean).clamp(0,1)

                ssim_val = ssim_metric(o_01, c_01).item()
                lpips_val = lpips_metric(o_01 * 2 - 1, c_01 * 2 - 1).item()
                isolated_gram_val = calculate_isolated_style_loss(target_img, style_img, model)

                # Log to CSV
                row_data = [
                    c_name, s_name, run_idx, output_filename,
                    round(study.best_params["alpha"], 4), round(study.best_params["beta"], 4), round(study.best_params["lr"], 4),
                    round(ssim_val, 4), round(lpips_val, 4), round(isolated_gram_val, 4), round(inf_latency, 2)
                ]

                with open(CSV_PATH, mode='a', newline='') as f:
                    csv.writer(f).writerow(row_data)

                completed_pairs.add(output_filename)

                # Critical GPU flushing to prevent OOM
                del content_img, style_img, target_img, content_features, style_features, study
                gc.collect()
                torch.cuda.empty_cache()

if __name__ == "__main__":
    main()

Mounted at /content/drive
-> Google Drive mounted successfully.
Using device: cuda
Targeting 5 Content images and 5 Style images.


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Processing: angel1024_composition_vii_run1.jpg


[W 2026-08-18 10:20:04,003] Trial 0 failed with parameters: {'alpha': 0.04593782538653204, 'beta': 825337.1656188808, 'lr': 1.8723436990700928} because of the following error: The value nan is not acceptable.
[W 2026-08-18 10:20:04,004] Trial 0 failed with value nan.


Processing: angel1024_composition_vii_run2.jpg
Processing: angel1024_composition_vii_run3.jpg
Processing: angel1024_feathers_run1.jpg
Processing: angel1024_feathers_run2.jpg
Processing: angel1024_feathers_run3.jpg
Processing: angel1024_la_muse_run1.jpg
Processing: angel1024_la_muse_run2.jpg
Processing: angel1024_la_muse_run3.jpg
Processing: angel1024_mosaic_run1.jpg
Processing: angel1024_mosaic_run2.jpg
Processing: angel1024_mosaic_run3.jpg
Processing: angel1024_wave_run1.jpg
Processing: angel1024_wave_run2.jpg
Processing: angel1024_wave_run3.jpg
Processing: berries1024_composition_vii_run1.jpg


[W 2026-08-18 11:02:14,894] Trial 2 failed with parameters: {'alpha': 0.044427024924647875, 'beta': 1881503.140356435, 'lr': 1.8631691786598654} because of the following error: The value nan is not acceptable.
[W 2026-08-18 11:02:14,895] Trial 2 failed with value nan.


Processing: berries1024_composition_vii_run2.jpg
Processing: berries1024_composition_vii_run3.jpg
Processing: berries1024_feathers_run1.jpg
Processing: berries1024_feathers_run2.jpg
Processing: berries1024_feathers_run3.jpg
Processing: berries1024_la_muse_run1.jpg
Processing: berries1024_la_muse_run2.jpg
Processing: berries1024_la_muse_run3.jpg
Processing: berries1024_mosaic_run1.jpg
Processing: berries1024_mosaic_run2.jpg
Processing: berries1024_mosaic_run3.jpg
Processing: berries1024_wave_run1.jpg
Processing: berries1024_wave_run2.jpg
Processing: berries1024_wave_run3.jpg
Processing: daisy1024_composition_vii_run1.jpg
Processing: daisy1024_composition_vii_run2.jpg
Processing: daisy1024_composition_vii_run3.jpg
Processing: daisy1024_feathers_run1.jpg
Processing: daisy1024_feathers_run2.jpg
Processing: daisy1024_feathers_run3.jpg


[W 2026-08-18 11:54:49,494] Trial 0 failed with parameters: {'alpha': 0.050860131019616596, 'beta': 372129.31983933685, 'lr': 1.964154834372819} because of the following error: The value nan is not acceptable.
[W 2026-08-18 11:54:49,495] Trial 0 failed with value nan.


Processing: daisy1024_la_muse_run1.jpg
Processing: daisy1024_la_muse_run2.jpg
Processing: daisy1024_la_muse_run3.jpg
Processing: daisy1024_mosaic_run1.jpg
Processing: daisy1024_mosaic_run2.jpg
Processing: daisy1024_mosaic_run3.jpg
Processing: daisy1024_wave_run1.jpg
Processing: daisy1024_wave_run2.jpg
Processing: daisy1024_wave_run3.jpg
Processing: mountains1024_composition_vii_run1.jpg
Processing: mountains1024_composition_vii_run2.jpg
Processing: mountains1024_composition_vii_run3.jpg
Processing: mountains1024_feathers_run1.jpg
Processing: mountains1024_feathers_run2.jpg
Processing: mountains1024_feathers_run3.jpg
Processing: mountains1024_la_muse_run1.jpg
Processing: mountains1024_la_muse_run2.jpg
Processing: mountains1024_la_muse_run3.jpg
Processing: mountains1024_mosaic_run1.jpg
Processing: mountains1024_mosaic_run2.jpg
Processing: mountains1024_mosaic_run3.jpg
Processing: mountains1024_wave_run1.jpg
Processing: mountains1024_wave_run2.jpg
Processing: mountains1024_wave_run3.jpg
P